# UPLOAD

In [54]:
from config import * 
import boto3
    
import os
    
from datetime import datetime
    
        
def backup_file():
    
    s3 = boto3.client("s3")
    
    backup_folder = datetime.now().strftime("Backup_%Y-%m-%d_%H-%M-%S")
    
    datetime.now()
    
    for file in os.listdir(LOCAL_FOLDER):
    
        path= os.path.join(LOCAL_FOLDER, file)
    
        if os.path.isfile(path):
    
            s3.upload_file(
                path,
                BUCKET_NAME,
                backup_folder + "/" + file
            )
    
    
            print(file, "Uploaded")
    print("Backup Completed")

# LIST BACKUP History

In [55]:
import boto3

from config import * 
def list_backup():
    s3 = boto3.client("s3")
    
    response = s3.list_objects_v2(Bucket=BUCKET_NAME)
    
    folders = set()
    
    for item in response.get ("Contents", []):
    
        folder = item["Key"].split("/")[0]
    
        folders.add(folder)
    print("Backup HISTORY ")
    
    for folder in sorted(folders):
        print(folder)

# # RESTORE 

In [56]:
import boto3

import os

from config import *

def restore():
    s3 = boto3.client("s3")
    
    backup_name = input("ENTER BACKUP Folder : ")
    
    response = s3.list_objects_v2(
        Bucket=BUCKET_NAME,
        Prefix=backup_name
    )
    
    
    os.makedirs(RESTORE_FOLDER, exist_ok=True)
    
    for item in response.get("Contents", []):
        filename = item["Key"].split("/")[-1]
        print(filename)
        if filename:
            destination = os.path.join(
                RESTORE_FOLDER,
                filename
            )
            s3.download_file(
                BUCKET_NAME,
                item["Key"],
                destination
            )
    
            print(filename, "Restored")
    print("Restore Completed ")

# DELETE ENTIRE BACKUP FOLDER FROM S3

In [57]:
import boto3
from config import *

def delete_backup_folder():
    s3 = boto3.client("s3")
    
    backup = input("BACKUP FOlder : ")
    
    response = s3.list_objects_v2(
        Bucket=BUCKET_NAME,
        Prefix=backup
    )

    
    
    for item in response.get("Contents", []):
        s3.delete_object(
            Bucket=BUCKET_NAME,
            Key=item["Key"]
        )
    
        print (item["Key"], "DELETED")
    
    print("BACKUP REMOVED")


# AUtomtic CLean LAST 3 Backup

In [61]:
import boto3
from config import *

def delete_3_backup():
    s3 = boto3.client("s3")
    response= s3.list_objects_v2(
        Bucket=BUCKET_NAME
    )   
    folders = set()
    
    for item in response.get("Contents", []):
        folders.add(item["Key"].split("/")[0])
    
    folders = sorted(folders)
    
    if len(folders) > 3:
        old = folders[:-3]
    
        for folder in old:
    
            response = s3.list_objects_v2(
                Bucket=BUCKET_NAME,
                Prefix=folder
            )
            for file in response.get("Contents", []):
                s3.delete_object(
                    Bucket=BUCKET_NAME,
                    Key = file["Key"]
                )
            print(folder , "DELETED")
    else:
        print("No Old Backup Found")



TypeError: 'str' object is not callable

In [ ]:
import backup
import restore
import list_backup
import delete_backup_folder
import delete_3_backup

while True:
    print("\n===========================")
    print("AWS CLOUD BACKUP SYSTEM ")
    print("\n===========================")
    print("1. Backup Files ")
    print("2. Restore Files ")
    print("3. List Backup ")
    print("4. Delete Backup  ")
    print("5. Delete Last 3 Backup ")
    print("6. EXIT")

    choice = input ("Enter the choice : ")

    if choice == "1":
        backup.backup_file()
    elif choice == "3":
        list_backup.list_backup()
    elif choice == "2":
        restore.restore()
    elif choice == "4":
        delete_backup_folder.delete_backup_folder()
    elif choice == "5":
        delete_3_backup.delete_3_backup()
    elif choice == "6":
        break
    else:
        print("\nEnter Valid Option from 1 to 6" )
        
        